# Multiple regression case study: Mario Kart

In this Lab, we'll consider Ebay auctions of a video game called *Mario Kart* for the Nintendo Wii. The outcome variable of interest is the total price of an auction, which is the highest bid plus the shipping cost. We will try to determine how total price is related to each characteristic in an auction while simultaneously controlling for other variables. For instance, all other characteristics held constant, are longer auctions associated with higher or lower prices? And, on average, how much more do buyers tend to pay for additional Wii wheels (plastic steering wheels that attach to the Wii controller) in auctions? Multiple regression will help us answer these and other questions.

## Dataset and the full model

First, we'll load the dataset, which includes a full week of auctions in early October 2009.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from plotnine import *

mariokart = pd.read_csv('https://raw.githubusercontent.com/clarkti5/math-209-data-sets/refs/heads/main/mariokart.csv')
mariokart.head()

### Exercise 1

- How many entries are in the dataset?
- Identify the variables and whether they are numerical or categorical.
- Create new indicator variables for the remaining categorical variables.

In [ ]:
# Run this code to create an indicator variable for `cond_new`
# that takes a value of 1 if the condition is`new`
# and takes a value of 0 if the condition is `used`.

mariokart['cond_new'] = np.where(mariokart['cond'] == 'new', 1, 0)

In [ ]:
# Fill in this code to create an indicator variable for `stock_photo`
# that takes a value of 1 if `stock_photo` is `yes`
# and takes a value of 0 if `stock_photo` is `no`.

mariokart['stock_photo_y'] = np.where(mariokart['???'] == '???', 1, 0)

### Exercise 2

- Run the code below to define the `least_squares_regression` function.

In [ ]:
import statsmodels.api as sm

def least_squares_regression(data, response_variable, predictor_variable):
  dataframe = data
  formula_string = response_variable + " ~ " + predictor_variable
  model = sm.formula.ols(formula = formula_string, data = dataframe)
  model_fitted = model.fit()

  print(model_fitted.summary())
  return model_fitted

- Then, use this function to fit a linear model that attempts to use `cond_new` to predict the price (stored in the `total_pr` variable).

In [ ]:
# Fill in the code!

least_squares_regression(mariokart, '???', '???')

- From the summary output, does `cond_new` appear to be a significant predictor of `total_pr`?

### Exercise 3

Recall, outliers in linear regression can affect the resulting model.

- Make a scatter plot of `cond_new` as a predictor of `total_pr`.
- Are there any outliers? If so, do those outliers appear to be influential?

In [ ]:
# Run this code to make the scatterplot.

(
    ggplot(mariokart) +
    aes(x = 'cond_new', y = 'total_pr') +
    geom_point() +
    geom_smooth(method = 'lm', se = False)
)

- It turns out, there were 2 auctions in the dataset that included *other* items besides just the *Mario Kart* game. Run the following code to remove these particularly extreme outliers.

In [ ]:
mariokart = mariokart[mariokart['total_pr'] < 100]

- Now, use `least_squares_regression` to refit the model.

In [ ]:
least_squares_regression(mariokart, 'total_pr', 'cond_new')

- With the outliers removed, does `cond_new` appear to be a significant predictor of `total_pr`?

### Exercise 4

Now use `least_squares_regression()` to fit a multiple regression model that uses

- `cond_new`
- `stock_photo`
- `duration`
- `wheels`
- `n_bids`
- `seller_rate`

to predict `total_pr`.

Then, refit the model after eliminating the variable with the largest p-value. Which model has a better adjusted $R^2$ value?

In [ ]:
# Run this code to fit the model.

least_squares_regression(mariokart, 'total_pr', 'cond_new + stock_photo + duration + wheels + n_bids + seller_rate')

In [ ]:
# Refit the model after eliminating the variable with the largest p-value

least_squares_regression(mariokart, 'total_pr', '???')

## Model selection

### Exercise 5

Continuing from Exercise 5, eliminate the variable with the largest p-value from the model and repeat this process until *all* of the variables are statistically significant predictors of `total_pr`.

Report the predictor variables remaining along with the model's adjusted $R^2$ value.

### Exercise 6

To be sure we have the most accurate model, start over with the full model from Exercise 5. Then, eliminate variables one at a time --- if eliminating the variable results in a higher adjusted $R^2$ value, do not include it in the model. Repeat this process until the adjusted $R^2$ value cannot be improved.

Report the predictor variables remaining along with the model's adjusted $R^2$ value.

In [ ]:
# Run this code to start again with the full model from Exercise 5.

least_squares_regression(mariokart, 'total_pr', 'cond_new + stock_photo + duration + wheels + n_bids + seller_rate')

### Exercise 7

- Fit a regression model that uses `cond_new`, `wheels`, and `seller_rate` to predict `total_pr`.

In [ ]:
# Fill in the code to fit a regression model that uses `cond_new`, `wheels`, and `seller_rate`
# to predict `total_pr`.

model = least_squares_regression(mariokart, 'total_pr', '???')

- Use this model to predict the total price of a Mario Kart game that is used, includes 1 wheel, and with seller rating 365.

- Compare this with the actually total price of the second entry (index 1) of the `mariokart` dataset. Are you surprised by the result? Why or why not?

### Exercise 8

Add columns `predicted_total_pr`, `residuals`, and `absolute_residuals` to `mariokart`.

In [ ]:
# Run this code to add the requested columns.

X = mariokart[['cond_new', 'wheels', 'seller_rate']]

mariokart['predicted_total_pr'] = model.predict(X)

mariokart['residuals'] = mariokart['total_pr'] - mariokart['predicted_total_pr']

mariokart['absolute_residuals'] = abs(mariokart['residuals'])

mariokart.head()

Compare the residual for the second entry (index 1) to what you found in Exercise 7.



## Checking model conditions with diagnostic plots

### Exercise 9

Check the nearly normal residuals condition using a histogram of the residuals.

In [ ]:
# Fill in the code to create a histogram of the residuals.

(
    ggplot(mariokart) +
    aes(x = '???') +
    geom_histogram()
)

### Exercise 10

Check the constant variability condition in the residuals with a scatterplot of the predicted values against the absolute value of the residuals.

In [ ]:
# Run this code to create the scatterplot to investigate the constant variability condition.

(
    ggplot(mariokart) +
    aes(x = 'predicted_total_pr', y = 'absolute_residuals') +
    geom_point() +
    geom_smooth(method = 'lm', se = False, color = 'red')
)

### Exercise 11

Check for independence by plotting the residuals in order of their corresponding auction (as indicated by their index in the dataset).

In [ ]:
# Run this code to create the plot and investigate the independence condition.

(
    ggplot(mariokart) +
    aes(x = mariokart.index, y = 'residuals') +
    geom_point() +
    geom_smooth(method = 'lm', se = False, color = 'red')
)

### Exercise 12

Now plot each predictor against the residuals to check that they hold a linear relationship. Report any concerns you may have.

In [ ]:
# Fill in the code to make a boxplot of the residuals by `cond_new`.
(
    ggplot(mariokart.replace([0,1], ["0", "1"])) +
    aes(x = '???', y = 'residuals') +
    geom_boxplot()
)

In [ ]:
# Fill in the code to make a scatterplot of `wheels` and the residuals.
(
    ggplot(mariokart) +
    aes(x = '???', y = 'residuals') +
    geom_point() +
    geom_smooth(se = False, color = 'red')
)

In [ ]:
# Fill in the code to make a scatterplot of `seller_rate` and the residuals.
(
    ggplot(mariokart) +
    aes(x = '???', y = 'residuals') +
    geom_point() +
    geom_smooth(se = False, color = 'red')
)

---

This lab was adapted by Timothy L. Clark, derivative of [OpenIntro Statistics by Diez, Çetinkaya-Rundel, and Barr](https://www.openintro.org/book/os/), released under [Creative Commons BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/deed.en) license.